
# Modelos de Linguagem: N-gramas e Embeddings (Python)

Este notebook demonstra:
1) Construção de um modelo **N-grama** (bigrama/trigrama) com geração de texto.
2) **Embeddings** de palavras via `gensim` (Word2Vec). Caso `gensim` não esteja disponível,
   é utilizado um **fallback** baseado em **coocorrência + SVD** para obter vetores de palavras.


In [28]:

# %% [setup] Importações e utilitários
import re
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt  # usar matplotlib, sem estilos/cores específicas

corpus_sentences = [
    "eu gosto de aprender machine learning com python",
    "a inteligência artificial está transformando a indústria moderna",
    "redes neurais profundas conseguem reconhecer imagens e sons",
    "python é uma linguagem popular para análise de dados",
    "estatística é fundamental para ciência de dados",
    "big data envolve processamento de grandes volumes de informação",
    "aprendizado de máquina supervisionado usa rótulos para treinar modelos",
    "aprendizado não supervisionado busca padrões ocultos",
    "modelos de linguagem predizem a próxima palavra em uma sequência",
    "word embeddings representam palavras como vetores densos",
    "o processamento de linguagem natural permite entender textos",
    "aprendizado por reforço envolve agentes que interagem com o ambiente",
    "deep learning revolucionou a visão computacional",
    "a regressão logística é usada em problemas de classificação",
    "a clusterização agrupa dados semelhantes sem rótulos",
    "o pandas facilita a manipulação de tabelas em python",
    "o numpy oferece operações eficientes com matrizes e vetores",
    "os gráficos do matplotlib ajudam a visualizar resultados",
    "o scikit learn oferece diversos algoritmos de aprendizado",
    "as transformadas wavelet podem ser aplicadas em sinais de áudio"
]

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    return [t for t in tokens if t.strip()]


## 1. Modelo N-grama (bigrama/trigrama)

In [29]:

# %% [ngrams] Construção de N-gramas
from collections import Counter

def build_ngrams(sentences, n=2, bos_token="<s>", eos_token="</s>"):
    ngram_counts = Counter()
    context_counts = Counter()
    for s in sentences:
        toks = [bos_token] * (n-1) + tokenize(s) + [eos_token]
        for i in range(len(toks) - n + 1):
            ngram = tuple(toks[i:i+n])
            context = ngram[:-1]
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts

def prob_ngram(ngram, ngram_counts, context_counts, alpha=1.0):
    context = ngram[:-1]
    # vocabulário aproximado a partir dos n-gramas
    V = len(set([w for ng in ngram_counts for w in ng]))
    return (ngram_counts[ngram] + alpha) / (context_counts[context] + alpha * V)

# Constrói bigramas e trigramas
bigram_counts, bigram_context = build_ngrams(corpus_sentences, n=2)
trigram_counts, trigram_context = build_ngrams(corpus_sentences, n=3)

print("Total de bigramas distintos:", len(bigram_counts))
print("Total de trigramas distintos:", len(trigram_counts))


Total de bigramas distintos: 173
Total de trigramas distintos: 177


### Geração de texto com N-gramas

In [30]:
bigram_counts
trigram_counts

Counter({('<s>', '<s>', 'eu'): 1,
         ('<s>', 'eu', 'gosto'): 1,
         ('eu', 'gosto', 'de'): 1,
         ('gosto', 'de', 'aprender'): 1,
         ('de', 'aprender', 'machine'): 1,
         ('aprender', 'machine', 'learning'): 1,
         ('machine', 'learning', 'com'): 1,
         ('learning', 'com', 'python'): 1,
         ('com', 'python', '</s>'): 1,
         ('<s>', '<s>', 'a'): 3,
         ('<s>', 'a', 'inteligência'): 1,
         ('a', 'inteligência', 'artificial'): 1,
         ('inteligência', 'artificial', 'está'): 1,
         ('artificial', 'está', 'transformando'): 1,
         ('está', 'transformando', 'a'): 1,
         ('transformando', 'a', 'indústria'): 1,
         ('a', 'indústria', 'moderna'): 1,
         ('indústria', 'moderna', '</s>'): 1,
         ('<s>', '<s>', 'redes'): 1,
         ('<s>', 'redes', 'neurais'): 1,
         ('redes', 'neurais', 'profundas'): 1,
         ('neurais', 'profundas', 'conseguem'): 1,
         ('profundas', 'conseguem', 'reconhecer')

In [31]:

# %% [generate] Geração de texto baseada em bigramas/trigramas
import random

def generate_text(ngram_counts, context_counts, n=2, max_len=20, bos_token="<s>", eos_token="</s>"):
    toks = [bos_token] * (n-1)
    for _ in range(max_len):
        context = tuple(toks[-(n-1):]) if n > 1 else tuple()
        candidates = []
        probs = []
        # coletar palavras que seguem este contexto
        vocab = set([ng[-1] for ng in ngram_counts if ng[:-1] == context])
        if not vocab:
            break
        total = 0.0
        for w in vocab:
            p = prob_ngram(context + (w,), ngram_counts, context_counts, alpha=1.0)
            candidates.append(w)
            probs.append(p)
            total += p
        probs = [p/total for p in probs]
        w = random.choices(candidates, weights=probs, k=1)[0]
        if w == eos_token:
            break
        toks.append(w)
    return " ".join([t for t in toks if t not in (bos_token, eos_token)])

print("Texto (bigramas):", generate_text(bigram_counts, bigram_context, n=2))
print("Texto (trigramas):", generate_text(trigram_counts, trigram_context, n=3))


Texto (bigramas): big data envolve agentes que interagem com matrizes e sons
Texto (trigramas): redes neurais profundas conseguem reconhecer imagens e sons


In [32]:
print("=== Exemplos com Bigramas ===")
for i in range(8):
    print(f"{i+1}.", generate_text(bigram_counts, bigram_context, n=2))

print("\n=== Exemplos com Trigramas ===")
for i in range(5):
    print(f"{i+1}.", generate_text(trigram_counts, trigram_context, n=3))

=== Exemplos com Bigramas ===
1. eu gosto de linguagem popular para treinar modelos de aprendizado não supervisionado usa rótulos para ciência de aprender machine learning
2. o processamento de máquina supervisionado usa rótulos
3. modelos
4. o numpy oferece diversos algoritmos de grandes volumes de grandes volumes de informação
5. as transformadas wavelet podem ser aplicadas em problemas de tabelas em sinais de informação
6. o processamento de dados semelhantes sem rótulos para análise de informação
7. word embeddings representam palavras como vetores densos
8. a inteligência artificial está transformando a manipulação de grandes volumes de informação

=== Exemplos com Trigramas ===
1. python é uma linguagem popular para análise de dados
2. aprendizado de máquina supervisionado usa rótulos para treinar modelos
3. o pandas facilita a manipulação de tabelas em python
4. word embeddings representam palavras como vetores densos
5. python é uma linguagem popular para análise de dados


### Perplexidade (avaliação simples)

In [6]:

# %% [perplexity] Cálculo de perplexidade
import math

def sentence_logprob(sentence, n, ngram_counts, context_counts, bos_token="<s>", eos_token="</s>", alpha=1.0):
    toks = [bos_token] * (n-1) + tokenize(sentence) + [eos_token]
    logp = 0.0
    N = 0
    for i in range(len(toks) - n + 1):
        ng = tuple(toks[i:i+n])
        p = prob_ngram(ng, ngram_counts, context_counts, alpha=alpha)
        logp += math.log(p + 1e-12)
        N += 1
    return logp, N

def perplexity(sentences, n, ngram_counts, context_counts, alpha=1.0):
    logp_sum, N_sum = 0.0, 0
    for s in sentences:
        lp, N = sentence_logprob(s, n, ngram_counts, context_counts, alpha=alpha)
        logp_sum += lp
        N_sum += N
    return math.exp(-logp_sum / max(N_sum, 1))

pp_bigram = perplexity(corpus_sentences, 2, bigram_counts, bigram_context, alpha=1.0)
pp_trigram = perplexity(corpus_sentences, 3, trigram_counts, trigram_context, alpha=1.0)
print(f"Perplexidade (bigramas): {pp_bigram:.2f}")
print(f"Perplexidade (trigramas): {pp_trigram:.2f}")


Perplexidade (bigramas): 22.49
Perplexidade (trigramas): 23.25


## 2. Embeddings de Palavras

In [9]:
import numpy as np
# %% [emb] Treino de embeddings com gensim (Word2Vec) se disponível; fallback coocorrência+SVD
def build_corpus_tokens(sentences):
    return [tokenize(s) for s in sentences]

tokenized = build_corpus_tokens(corpus_sentences)
EMBEDDINGS_BACKEND = None

vocab_words = []
def backend_word2vec(tokenized):
    from gensim.models import Word2Vec
    model = Word2Vec(tokenized, vector_size=50, window=3, min_count=1, workers=1, sg=1, epochs=200)
    def get_vec(w):
        return model.wv[w]
    words = list(model.wv.key_to_index.keys())
    return "gensim_word2vec", get_vec, words

def backend_cooc_svd(tokenized):
    # Matriz de coocorrência por janela
    window = 2
    vocab = sorted({w for sent in tokenized for w in sent})
    idx = {w:i for i,w in enumerate(vocab)}
    C = np.zeros((len(vocab), len(vocab)), dtype=float)
    for sent in tokenized:
        for i, w in enumerate(sent):
            wi = idx[w]
            left = max(0, i-window); right = min(len(sent), i+window+1)
            for j in range(left, right):
                if j == i:
                    continue
                wj = idx[sent[j]]
                C[wi, wj] += 1.0
    total = C.sum()
    sum_w = C.sum(axis=1, keepdims=True) + 1e-12
    sum_c = C.sum(axis=0, keepdims=True) + 1e-12
    # PPMI
    import numpy as np
    with np.errstate(divide="ignore"):
        PMI = np.log((C * total) / (sum_w * sum_c))
    PMI[np.isinf(PMI)] = 0.0
    PPMI = np.maximum(PMI, 0.0)
    # SVD
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(n_components=50, random_state=42)
    W = svd.fit_transform(PPMI)
    def get_vec(w):
        return W[idx[w]]
    return "cooc_svd", get_vec, vocab

# Tenta gensim, senão fallback
try:
    EMBEDDINGS_BACKEND, get_vector, vocab_words = backend_word2vec(tokenized)
    print("Embeddings treinados com gensim.Word2Vec.")
except Exception as e:
    print("gensim não disponível; usando fallback coocorrência+SVD.", e)
    EMBEDDINGS_BACKEND, get_vector, vocab_words = backend_cooc_svd(tokenized)

print("Backend de embeddings:", EMBEDDINGS_BACKEND)
print("Vocabulário (amostra):", vocab_words[:10], "...")


Embeddings treinados com gensim.Word2Vec.
Backend de embeddings: gensim_word2vec
Vocabulário (amostra): ['de', 'eu', 'learning', 'python', 'linguagem', 'machine', 'aprender', 'dados', 'uma', 'é'] ...


### Similaridade semântica

In [10]:

# %% [similarity] Vizinhos mais semelhantes (cosseno)
from numpy.linalg import norm
import numpy as np

def cosine_sim(a, b):
    a = np.asarray(a); b = np.asarray(b)
    return float(np.dot(a, b) / ((norm(a)+1e-12)*(norm(b)+1e-12)))

def most_similar(query, topn=5):
    if query not in vocab_words:
        return []
    qv = get_vector(query)
    sims = []
    for w in vocab_words:
        if w == query:
            continue
        sims.append((w, cosine_sim(qv, get_vector(w))))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

for q in ["python", "dados", "linguagem", "learning"]:
    print(f"\nMais semelhantes a '{q}':")
    for w, s in most_similar(q, topn=5):
        print(f"  {w:>12s}  {s:.3f}")



Mais semelhantes a 'python':
            de  0.837
      capturam  0.820
         entre  0.817
    semânticas  0.799
           uma  0.787

Mais semelhantes a 'dados':
            de  0.781
           uma  0.772
      capturam  0.761
     linguagem  0.742
    semânticas  0.734

Mais semelhantes a 'linguagem':
      learning  0.830
           uma  0.819
            de  0.818
            eu  0.800
      capturam  0.799

Mais semelhantes a 'learning':
            de  0.873
           uma  0.860
         entre  0.846
      capturam  0.830
     linguagem  0.830


### Visualização 2D dos embeddings (PCA)

In [ ]:

# %% [viz] Projeção PCA em 2D (um gráfico, sem especificar cores)
from sklearn.decomposition import PCA

def build_matrix(words):
    X = []
    keep = []
    for w in words:
        try:
            vec = get_vector(w)
            X.append(vec)
            keep.append(w)
        except Exception:
            pass
    return np.array(X), keep

sample_words = sorted(set(sum(tokenized, [])))
X, keep = build_matrix(sample_words)

if len(X) >= 2:
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(X)

    plt.figure(figsize=(8, 6))
    plt.scatter(X2[:, 0], X2[:, 1])
    for i, w in enumerate(keep):
        plt.annotate(w, (X2[i, 0], X2[i, 1]), fontsize=9)
    plt.title("Projeção PCA dos Embeddings (2D)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()
else:
    print("Poucas palavras para visualizar.")



## 3. Exercícios Sugeridos

1. **Geração de Texto (N-gramas)**  
   - Aumente `n` para 3 (trigramas) e compare a fluência do texto gerado.
   - Teste diferentes valores de `alpha` (suavização de Laplace) e observe o impacto na perplexidade.

2. **Embeddings**  
   - Se `gensim` estiver disponível, varie `window`, `vector_size` e `epochs` e observe os vizinhos mais semelhantes.
   - No fallback (coocorrência+SVD), teste outras janelas e número de componentes do SVD.

3. **Visualização**  
   - Experimente `t-SNE` (`sklearn.manifold.TSNE`) para visualizar os vetores em 2D e compare com PCA.

4. **Corpora maiores**  
   - Substitua `corpus_sentences` por um conjunto de sentenças maior (ex.: Wikipedia em português, notícias)
     para observar melhor o ganho de semântica nos embeddings.
